# CSC4792 Data Mining and Warehousing
## Group 26 — Mpongwe Town Council Dataset

This notebook documents the extraction, cleaning, preprocessing, validation, and export of related **Mpongwe Town Council datasets** from official council PDFs. It preserves the completed 2025 CDF project pipeline and adds a multi-year procurement-plan dataset.

The final dataset is exported as a pipe-separated (`|`) CSV in accordance with the CSC4792 mini-project requirements.

## 1. Introduction

The objective of this notebook is to transform official Mpongwe Town Council documents into reusable, traceable tabular datasets. Part A retains the completed **Mpongwe Constituency 2025 Approved and Not Approved CDF Projects** dataset. Part B adds procurement-plan records from 2023, 2024, and 2025.

The workflow is:

1. Extract text and tables from the source PDF.
2. Segment the document into individual project records.
3. Recover sector, ward, and approval information.
4. Reconstruct clean project descriptions from PDF table cells.
5. Standardise rejection reasons.
6. Validate the resulting records.
7. Export the final dataset using the required pipe (`|`) separator.
8. Inspect and pair procurement-plan pages by their original source row numbers.
9. Normalize only reliably extractable procurement fields, validate them, and export a second pipe-separated dataset.

## 2. Data Source and Setup

The sources are official Mpongwe Town Council PDFs: a 2025 approved and not approved CDF project list and annual procurement plans for 2023, 2024, and 2025.

**Expected local project structure**

```text
CSC4792_Group26_Mpongwe/
├── data/
│   ├── raw/
│   │   └── mpongwe_cdf_projects_2025.pdf
│   └── processed/
├── notebooks/
│   └── group26_mpongwe_dataset_creation.ipynb
└── documents/
```

The notebook assumes it is stored inside the `notebooks/` directory.

In [1]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import pdfplumber

# Support execution from either the repository root or notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pdf_path = PROJECT_ROOT / "data/raw/mpongwe_cdf_projects_2025.pdf"
output_path = PROJECT_ROOT / "data/processed/db-unza26-csc4792-mpongwe_cdf_projects.csv"

source_url = (
    "http://www.mpongwecouncil.gov.zm/wp-content/uploads/2025/11/"
    "APPROVED-AND-UNAPPROVED-COMMUNITY-PROJECTS-2025.pdf"
)

EXPECTED_PROJECTS = 84

## 3. PDF Text Extraction

The PDF contains tables whose layouts vary between pages. Raw page text is first extracted so that numbered project records can be identified in sequence.

In [2]:
all_text = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            all_text.append(text)

full_text = "\n".join(all_text)

print("Characters extracted:", len(full_text))

Characters extracted: 9425


### 3.1 Project Record Segmentation

A naive split on lines beginning with a number initially produces false records because some descriptions contain numbered items such as **"3 toilets"**. The records are therefore repaired by enforcing the expected sequential project numbering.

In [3]:
records = re.split(r"(?m)(?=^\d+\s+)", full_text)

records = [
    record.strip()
    for record in records
    if re.match(r"^\d+\s+", record.strip())
]

clean_records = []
expected_number = 1

for record in records:
    match = re.match(r"^(\d+)\s+", record.strip())
    if not match:
        continue

    number = int(match.group(1))

    if number == expected_number:
        clean_records.append(record.strip())
        expected_number += 1
    elif clean_records:
        # Treat out-of-sequence numbered text as a continuation
        # of the previous project description.
        clean_records[-1] += "\n" + record.strip()

project_nums = []

for record in clean_records:
    match = re.match(r"^(\d+)\s+", record)
    if match:
        project_nums.append(int(match.group(1)))

missing_numbers = [
    n for n in range(1, max(project_nums) + 1)
    if n not in project_nums
]

duplicates = [
    number
    for number, count in Counter(project_nums).items()
    if count > 1
]

print("Total cleaned project records:", len(clean_records))
print("First project:", project_nums[0])
print("Last project:", project_nums[-1])
print("Missing project numbers:", missing_numbers)
print("Duplicate project numbers:", duplicates)

assert len(clean_records) == EXPECTED_PROJECTS
assert missing_numbers == []
assert duplicates == []

Total cleaned project records: 84
First project: 1
Last project: 84
Missing project numbers: []
Duplicate project numbers: []


## 4. Sector, Ward, and Approval Status Extraction

Known sector and ward labels are used as anchors because the raw PDF text interleaves information from multiple table columns.

More specific multi-word sector labels are checked before shorter labels to avoid misclassification. A small number of values require explicit recovery rules because the PDF extraction separates words across table columns.

In [4]:
known_sectors = [
    "Education/Health",
    "Water and Sanitation",
    "Road/ Transport",
    "Health & Road",
    "Local Government",
    "Communication",
    "Infrastructure",
    "Agriculture",
    "Information",
    "Transport",
    "Security",
    "Economic",
    "Education",
    "Health"
]

known_wards = [
    "Kasamba",
    "Kanyenda",
    "Ntanda",
    "Kashiba",
    "Luswishi",
    "Mpongwe Central",
    "Kasonga",
    "Ipumbu",
    "Ibenga",
    "Nampamba",
    "Chisapa",
    "Musofu",
    "Mikata",
    "Munkumpu",
    "Kalweo",
    "All Wards"
]

In [5]:
parsed_records = []

for record in clean_records:
    lines = [line.strip() for line in record.split("\n") if line.strip()]
    first_line = lines[0]

    match = re.match(r"^(\d+)\s+(.*)", first_line)
    if not match:
        continue

    project_no = match.group(1)
    full_record = " ".join(lines)

    sector = None
    ward = None

    # Sector extraction
    for sector_name in known_sectors:
        if sector_name.lower() in full_record.lower():
            sector = sector_name
            break

    # Recover sectors affected by PDF column splitting
    if sector is None:
        if project_no == "31":
            sector = "Local Government"
        elif project_no in {"55", "65", "78"}:
            sector = "Water and Sanitation"

    # Ward extraction
    for ward_name in known_wards:
        if ward_name.lower() in full_record.lower():
            ward = ward_name
            break

    # Mpongwe Central is sometimes split across columns
    if ward is None:
        text_lower = full_record.lower()
        if "mpongwe" in text_lower and "central" in text_lower:
            ward = "Mpongwe Central"

    # Approval status
    if project_no == "33":
        approval_status = "Partially Approved"
    elif "approved" in full_record.lower():
        approval_status = "Approved"
    else:
        approval_status = "Not Approved"

    parsed_records.append({
        "project_no": project_no,
        "raw_record": full_record,
        "sector": sector,
        "ward": ward,
        "approval_status": approval_status
    })

parsed_df = pd.DataFrame(parsed_records)

print("Parsed records:", len(parsed_df))
print("Missing wards:", parsed_df["ward"].isna().sum())
print("Missing sectors:", parsed_df["sector"].isna().sum())
print("\nApproval status counts:")
print(parsed_df["approval_status"].value_counts())

assert len(parsed_df) == EXPECTED_PROJECTS
assert parsed_df["ward"].isna().sum() == 0

Parsed records: 84
Missing wards: 0
Missing sectors: 4

Approval status counts:
approval_status
Not Approved          71
Approved              12
Partially Approved     1
Name: count, dtype: int64


## 5. Rejection Reason Extraction

Most rejected projects use recurring reason phrases such as **Inadequate Funding** or **Not the first priority**. A few projects contain unique explanations, so those are handled explicitly.

Approved projects intentionally have no rejection reason.

In [6]:
def extract_reason(row):
    project_no = str(row["project_no"])
    text = row["raw_record"].lower()
    status = row["approval_status"]

    if status == "Approved":
        return None

    # Special cases preserved from the source
    if project_no == "33":
        return "Motor bikes not approved; bicycles approved instead"

    if project_no == "32":
        return (
            "Not the first priority; hospital was funded in the "
            "previous year with medical equipment"
        )

    if project_no == "36":
        return (
            "A modern police station was already under construction; "
            "project falls under the Ministry of Infrastructure"
        )

    if project_no == "37":
        return (
            "No documentation for transfer of property from "
            "Provincial Administration"
        )

    if project_no == "62":
        return "Not amongst the recommended projects in the CDF Guidelines"

    # General recurring reasons
    if "inadequate funding" in text:
        return "Inadequate Funding"

    if (
        "not the first priority" in text
        or "not on the first" in text
        or ("not the first" in text and "priority" in text)
    ):
        return "Not the first priority"

    if "not a priority" in text:
        return "Not a priority"

    return "Reason requires manual review"


parsed_df["reason_not_approved"] = parsed_df.apply(
    extract_reason,
    axis=1
)

print(parsed_df["reason_not_approved"].value_counts(dropna=False))

manual_review_count = (
    parsed_df["reason_not_approved"]
    .eq("Reason requires manual review")
    .sum()
)

missing_nonapproved_reasons = parsed_df[
    (parsed_df["approval_status"] != "Approved")
    & (parsed_df["reason_not_approved"].isna())
]

print("\nManual-review reasons:", manual_review_count)
print("Non-approved records without a reason:", len(missing_nonapproved_reasons))

assert manual_review_count == 0
assert len(missing_nonapproved_reasons) == 0

reason_not_approved
Inadequate Funding                                                                                            41
Not the first priority                                                                                        25
NaN                                                                                                           12
Not a priority                                                                                                 1
Not the first priority; hospital was funded in the previous year with medical equipment                        1
Motor bikes not approved; bicycles approved instead                                                            1
A modern police station was already under construction; project falls under the Ministry of Infrastructure     1
No documentation for transfer of property from Provincial Administration                                       1
Not amongst the recommended projects in the CDF Guidelines                  



Manual-review reasons: 0
Non-approved records without a reason: 0


## 6. Project Description Reconstruction

Raw text extraction mixes sector, ward, approval status, and project description together. To obtain cleaner project names, descriptions are reconstructed directly from PDF table cells.

The PDF uses multiple table layouts, so the extractor searches description text across columns 1, 2, and 3. Confirmed duplicated extraction fragments are then removed using targeted corrections.

In [7]:
project_descriptions = {}

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            if not table:
                continue

            # Ignore small auxiliary tables that contain no numbered projects.
            has_project_rows = any(
                row
                and len(row) > 0
                and row[0]
                and str(row[0]).strip().isdigit()
                for row in table
            )

            if not has_project_rows:
                continue

            current_project_no = None
            current_parts = []

            for row in table:
                if not row:
                    continue

                project_no = row[0] if len(row) > 0 else None

                # Description text may occur in columns 1, 2, or 3.
                description_part = None

                for col in [1, 2, 3]:
                    if len(row) > col and row[col]:
                        value = str(row[col]).replace("\n", " ").strip()
                        if value:
                            description_part = value
                            break

                if project_no and str(project_no).strip().isdigit():
                    if current_project_no is not None:
                        project_descriptions[current_project_no] = (
                            " ".join(current_parts).strip()
                        )

                    current_project_no = str(project_no).strip()
                    current_parts = []

                    if description_part:
                        current_parts.append(description_part)

                elif current_project_no is not None and description_part:
                    current_parts.append(description_part)

            if current_project_no is not None:
                project_descriptions[current_project_no] = (
                    " ".join(current_parts).strip()
                )

print("Descriptions extracted:", len(project_descriptions))

Descriptions extracted: 84


In [8]:
# Targeted corrections remove duplicated fragments introduced by
# inconsistent PDF table layouts. The wording of the source is otherwise preserved.
description_corrections = {
    "4": "Construction of a Community School at Fikola area.",
    "6": "Construction of a health post at Bwandaila in Milonfi",
    "11": "Construction of 1 staff house at Nkulumashiba School",
    "16": "Construction of 1x3 CRB and 2 staff houses in Twikatane",
    "20": "Construction of 1x3 CRB at Ikula Community School",
    "23": "Construction of 1 staff house at Mfulabunga Clinic",
    "31": "Construction of Parliamentary Constituency Office",
    "36": "Construction of a modern police station",
    "38": "Purchase of X- Ray and Scanner at Mpongwe Mission Hospital",
    "43": "Construction of Agricultural staff house at Mushiwe",
    "45": "Construction of 1x3 CRB at Nkulushu Primary School",
    "52": "Construction of ablution block at St.theresa's Secondary school",
    "56": "Construction of Muchindushi bridge in Chowa zone",
    "58": "Completion of John Chawa clinic in Chawama zone",
    "61": "Grading of road connecting Kantolo to Chisapa",
    "73": "Upgrading of Mikata Rural Health Center to mini hospital",
    "76": "Construction of a health post and grading of feeder road in Chitabale"
}

project_descriptions.update(description_corrections)

blank_descriptions = [
    project_no
    for project_no, description in project_descriptions.items()
    if not description.strip()
]

print("Total descriptions:", len(project_descriptions))
print("Blank descriptions:", blank_descriptions)

assert len(project_descriptions) == EXPECTED_PROJECTS
assert blank_descriptions == []

Total descriptions: 84
Blank descriptions: []


## 7. Final Dataset Construction

The reconstructed descriptions are combined with the extracted sector, ward, approval status, rejection reason, and source metadata.

### Dataset fields

| Column | Description |
|---|---|
| `project_id` | Unique identifier created for each project |
| `sector` | Project sector reported in the source |
| `ward` | Ward associated with the project |
| `project_name` | Reconstructed project name/description |
| `approval_status` | Approved, Not Approved, or Partially Approved |
| `reason_not_approved` | Reason given when a project was not fully approved |
| `year` | Project year |
| `source_document` | Name of the source document |
| `source_url` | Official source URL |

In [9]:
parsed_df["project_name"] = parsed_df["project_no"].map(project_descriptions)

parsed_df["project_id"] = parsed_df["project_no"].apply(
    lambda x: f"CDF2025-{int(x):03d}"
)

parsed_df["year"] = 2025
parsed_df["source_document"] = (
    "Mpongwe 2025 Approved and Unapproved CDF Community Projects"
)
parsed_df["source_url"] = source_url

final_df = parsed_df[
    [
        "project_id",
        "sector",
        "ward",
        "project_name",
        "approval_status",
        "reason_not_approved",
        "year",
        "source_document",
        "source_url"
    ]
].copy()

final_df.head()

,project_id,sector,ward,project_name,approval_status,reason_not_approved,year,source_document,source_url
0,CDF2025-001,Education,Kasamba,Construction of a 1X4 CRB at Kasamba Secondary...,Not Approved,Inadequate Funding,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
1,CDF2025-002,Health,Kasamba,Construction of 1 staff house and completion o...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
2,CDF2025-003,Education,Kasamba,Construction of 1 staff house; purchase of 80 ...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
3,CDF2025-004,Education,Kasamba,Construction of a Community School at Fikola a...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
4,CDF2025-005,Education,Kasamba,Construction of 1 staff house and purchase of ...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...


## 8. Data Validation

The final dataset is checked for record count, duplicate identifiers, duplicate rows, missing project names, approval-status consistency, and missing values.

Four sector values remain missing because the extracted source does not preserve a reliable sector for those records. They are intentionally left missing rather than inferred.

In [10]:
print("Total rows:", len(final_df))
print("Duplicate project IDs:", final_df["project_id"].duplicated().sum())
print("Duplicate full rows:", final_df.duplicated().sum())
print("Missing project names:", final_df["project_name"].isna().sum())

print("\nApproval status counts:")
print(final_df["approval_status"].value_counts())

print("\nWard counts:")
print(final_df["ward"].value_counts())

print("\nMissing values:")
print(final_df.isnull().sum())

assert len(final_df) == EXPECTED_PROJECTS
assert final_df["project_id"].duplicated().sum() == 0
assert final_df.duplicated().sum() == 0
assert final_df["project_name"].isna().sum() == 0
assert final_df["ward"].isna().sum() == 0
assert final_df["approval_status"].value_counts().sum() == EXPECTED_PROJECTS

Total rows: 84
Duplicate project IDs: 0
Duplicate full rows: 0
Missing project names: 0

Approval status counts:
approval_status
Not Approved          71
Approved              12
Partially Approved     1
Name: count, dtype: int64

Ward counts:
ward
Mpongwe Central    11
Kasamba             9
Ntanda              6
Ibenga              6
Kanyenda            5
Luswishi            5
Kasonga             5
Nampamba            5
Chisapa             5
Mikata              5
Munkumpu            5
Kalweo              5
Kashiba             4
Musofu              4
Ipumbu              3
All Wards           1
Name: count, dtype: int64

Missing values:
project_id              0
sector                  4
ward                    0
project_name            0
approval_status         0
reason_not_approved    12
year                    0
source_document         0
source_url              0
dtype: int64


In [11]:
print("First project ID:", final_df["project_id"].iloc[0])
print("Last project ID:", final_df["project_id"].iloc[-1])

First project ID: CDF2025-001
Last project ID: CDF2025-084


## 9. Final Dataset Export

The final dataset is exported as a CSV using the required pipe (`|`) delimiter and the CSC4792 naming convention.

In [12]:
final_df.to_csv(
    output_path,
    sep="|",
    index=False
)

print("Dataset exported to:")
print(output_path)

Dataset exported to:
C:\Users\USER\Documents\Uni Study Notes\4th Year\csc 4792\Gassignment\CSC4792_Group26_Mpongwe\data\processed\db-unza26-csc4792-mpongwe_cdf_projects.csv


In [13]:
# Read the exported dataset back in to verify that it was written correctly.
check_df = pd.read_csv(
    output_path,
    sep="|"
)

print("Exported shape:", check_df.shape)
print("Duplicate IDs:", check_df["project_id"].duplicated().sum())
print("Missing project names:", check_df["project_name"].isna().sum())

print("\nApproval counts:")
print(check_df["approval_status"].value_counts())

check_df.head()

Exported shape: (84, 9)
Duplicate IDs: 0
Missing project names: 0

Approval counts:
approval_status
Not Approved          71
Approved              12
Partially Approved     1
Name: count, dtype: int64


,project_id,sector,ward,project_name,approval_status,reason_not_approved,year,source_document,source_url
0,CDF2025-001,Education,Kasamba,Construction of a 1X4 CRB at Kasamba Secondary...,Not Approved,Inadequate Funding,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
1,CDF2025-002,Health,Kasamba,Construction of 1 staff house and completion o...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
2,CDF2025-003,Education,Kasamba,Construction of 1 staff house; purchase of 80 ...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
3,CDF2025-004,Education,Kasamba,Construction of a Community School at Fikola a...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...
4,CDF2025-005,Education,Kasamba,Construction of 1 staff house and purchase of ...,Not Approved,Not the first priority,2025,Mpongwe 2025 Approved and Unapproved CDF Commu...,http://www.mpongwecouncil.gov.zm/wp-content/up...


# PART B - PROCUREMENT DATASET

## 8. Procurement Source Discovery

The official Mpongwe Town Council publications page links to annual procurement plans for 2023, 2024, and 2025. The files were saved locally under `data/raw/procurement/` using stable filenames. All three PDFs contain machine-readable spreadsheet tables. The 2023 plan is split across two horizontal pages; the 2025 plan is split across six pages, with left, right, and mostly empty continuation segments. The 2024 plan is one page. Repeated spreadsheet headers and page numbers are excluded by retaining only numbered procurement rows.

The source PDFs visibly clip some 2023 and 2025 description and reference-number cells at column boundaries. Those values are kept missing rather than reconstructed or guessed.

In [14]:
procurement_dir = PROJECT_ROOT / "data/raw/procurement"
procurement_output_path = PROJECT_ROOT / "data/processed/db-unza26-csc4792-mpongwe_procurement.csv"

procurement_sources = {
    2023: {
        "file": procurement_dir / "mpongwe_procurement_plan_2023.pdf",
        "url": "https://www.mpongwecouncil.gov.zm/wp-content/uploads/2023/11/APP-MPONGWE-TOWN-COUNCIL-1.pdf",
    },
    2024: {
        "file": procurement_dir / "mpongwe_procurement_plan_2024.pdf",
        "url": "https://www.mpongwecouncil.gov.zm/wp-content/uploads/2024/02/MPONGWE-TOWN-COUNCIL-APP-2024-FINAL.pdf",
    },
    2025: {
        "file": procurement_dir / "mpongwe_procurement_plan_2025.pdf",
        "url": "http://www.mpongwecouncil.gov.zm/wp-content/uploads/2025/12/ANNUAL-PROCUREMENT-PLAN-2025.pdf",
    },
}

for source in procurement_sources.values():
    assert source["file"].exists(), f"Missing raw source: {source['file']}"

PROCUREMENT_COLUMNS = [
    "procurement_id", "year", "procurement_class", "unspsc",
    "description", "reference_number", "unit_of_measure", "quantity",
    "source_of_funds", "procurement_method", "publication_date",
    "award_date", "start_date", "estimated_budget_zmw", "comments",
    "source_document", "source_url",
]

## 9. Procurement Extraction

The extraction uses `pdfplumber` tables and the printed spreadsheet row number as the join key for horizontally split pages. It does not use page headers, totals, or page numbers as records. The generated identifier (`APP<year>-<source row>`) is stable, unique, and traceable to the original plan row.

In [15]:
def clean_text(value):
    if value is None:
        return pd.NA
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value if value else pd.NA


def numbered_rows(table):
    rows = {}
    for row in table:
        row_number = clean_text(row[0])
        if row_number is not pd.NA and str(row_number).isdigit() and int(row_number) >= 11:
            rows[int(row_number)] = row
    return rows


def source_date(value):
    value = clean_text(value)
    if value is pd.NA:
        return pd.NA
    parsed = pd.to_datetime(value, dayfirst=True, errors="coerce")
    return parsed.date().isoformat() if not pd.isna(parsed) else pd.NA


def source_number(value):
    value = clean_text(value)
    if value is pd.NA:
        return pd.NA
    value = str(value).replace(",", "")
    try:
        return float(value)
    except ValueError:
        return pd.NA


def extract_reference(value, year):
    value = clean_text(value)
    if value is pd.NA:
        return pd.NA
    match = re.search(rf"MTC(?:[-/][A-Za-z0-9]+)+/{str(year)[-2:]}", str(value), re.IGNORECASE)
    return match.group(0).upper() if match else pd.NA


def make_record(year, row_number, **values):
    return {
        "procurement_id": f"APP{year}-{row_number:03d}",
        "year": year,
        "source_document": procurement_sources[year]["file"].name,
        "source_url": procurement_sources[year]["url"],
        **values,
    }


procurement_records = []

with pdfplumber.open(procurement_sources[2023]["file"]) as pdf:
    left_2023 = numbered_rows(pdf.pages[0].extract_table())
    right_2023 = numbered_rows(pdf.pages[1].extract_table())

for row_number in sorted(left_2023):
    left = left_2023[row_number]
    right = right_2023[row_number]
    procurement_records.append(make_record(
        2023, row_number,
        procurement_class=clean_text(left[1]),
        unspsc=clean_text(left[2]),
        description=pd.NA,
        reference_number=extract_reference(left[5], 2023),
        unit_of_measure=clean_text(left[7]),
        quantity=source_number(left[8]),
        source_of_funds=clean_text(right[5]),
        procurement_method=clean_text(left[10]),
        publication_date=source_date(left[11]),
        award_date=source_date(left[12]),
        start_date=source_date(right[1]),
        estimated_budget_zmw=pd.NA,
        comments=clean_text(right[4]),
    ))

with pdfplumber.open(procurement_sources[2024]["file"]) as pdf:
    rows_2024 = numbered_rows(pdf.pages[0].extract_table())

for row_number, row in sorted(rows_2024.items()):
    procurement_records.append(make_record(
        2024, row_number,
        procurement_class=clean_text(row[1]),
        unspsc=clean_text(row[2]),
        description=clean_text(row[4]),
        reference_number=extract_reference(row[5], 2024),
        unit_of_measure=clean_text(row[6]),
        quantity=source_number(row[7]),
        source_of_funds=clean_text(row[8]),
        procurement_method=pd.NA,
        publication_date=source_date(row[9]),
        award_date=pd.NA,
        start_date=source_date(row[10]),
        estimated_budget_zmw=pd.NA,
        comments=clean_text(row[12]),
    ))

with pdfplumber.open(procurement_sources[2025]["file"]) as pdf:
    left_2025 = {**numbered_rows(pdf.pages[0].extract_table()), **numbered_rows(pdf.pages[1].extract_table())}
    right_2025 = {**numbered_rows(pdf.pages[2].extract_table()), **numbered_rows(pdf.pages[3].extract_table())}

for row_number in sorted(left_2025):
    left = left_2025[row_number]
    right = right_2025[row_number]
    procurement_records.append(make_record(
        2025, row_number,
        procurement_class=clean_text(left[1]),
        unspsc=clean_text(left[2]),
        description=pd.NA,
        reference_number=pd.NA,
        unit_of_measure=clean_text(left[7]),
        quantity=source_number(left[8]),
        source_of_funds=clean_text(right[1]),
        procurement_method=clean_text(right[3]),
        publication_date=source_date(right[4]),
        award_date=source_date(right[5]),
        start_date=source_date(right[6]),
        estimated_budget_zmw=source_number(left[11]),
        comments=clean_text(right[9]),
    ))

procurement_df = pd.DataFrame(procurement_records, columns=PROCUREMENT_COLUMNS)
for column in [
    "procurement_id", "procurement_class", "unspsc", "description",
    "reference_number", "unit_of_measure", "source_of_funds",
    "procurement_method", "publication_date", "award_date", "start_date",
    "comments", "source_document", "source_url",
]:
    procurement_df[column] = procurement_df[column].astype("string")
procurement_df["year"] = procurement_df["year"].astype("Int64")
procurement_df["quantity"] = procurement_df["quantity"].astype("Float64")
procurement_df["estimated_budget_zmw"] = procurement_df["estimated_budget_zmw"].astype("Float64")

procurement_df.head()

,procurement_id,year,procurement_class,unspsc,description,reference_number,unit_of_measure,quantity,source_of_funds,procurement_method,publication_date,award_date,start_date,estimated_budget_zmw,comments,source_document,source_url
0,APP2023-011,2023,goods,22101500,<NA>,<NA>,Each,1.0,CDF,obn,2023-06-09,2023-07-17,2023-08-23,<NA>,<NA>,mpongwe_procurement_plan_2023.pdf,https://www.mpongwecouncil.gov.zm/wp-content/u...
1,APP2023-012,2023,goods,25101601,<NA>,<NA>,Each,1.0,CDF,obn,2023-06-09,2023-07-17,2023-08-25,<NA>,<NA>,mpongwe_procurement_plan_2023.pdf,https://www.mpongwecouncil.gov.zm/wp-content/u...
2,APP2023-013,2023,goods,25111505,<NA>,<NA>,Each,1.0,CDF,obn,2023-06-09,2023-07-17,2023-08-24,<NA>,<NA>,mpongwe_procurement_plan_2023.pdf,https://www.mpongwecouncil.gov.zm/wp-content/u...
3,APP2023-014,2023,goods,22101511,<NA>,<NA>,Each,1.0,CDF,obn,2023-06-09,2023-07-17,2023-08-23,<NA>,<NA>,mpongwe_procurement_plan_2023.pdf,https://www.mpongwecouncil.gov.zm/wp-content/u...
4,APP2023-015,2023,works,30120000,<NA>,<NA>,sV aMrpioounsgwe ward,<NA>,CDF,sb,2023-06-08,2023-07-05,2023-07-11,<NA>,<NA>,mpongwe_procurement_plan_2023.pdf,https://www.mpongwecouncil.gov.zm/wp-content/u...


## 10. Procurement Cleaning and Preprocessing

Whitespace and line breaks are normalized, source dates are converted to ISO `YYYY-MM-DD`, and numeric quantities/budgets are converted only when the source supplies a valid numeric value. Missing values remain missing. Equivalent date and funding fields are normalized across plans only when clearly labelled.

## 11. Procurement Validation

The checks below confirm expected plan-row counts, unique generated IDs, no duplicate full records, source-year consistency, complete source provenance, and the absence of header/page-number rows.

In [16]:
expected_rows_by_year = {2023: 47, 2024: 37, 2025: 63}
assert procurement_df.groupby("year").size().to_dict() == expected_rows_by_year
assert len(procurement_df) == sum(expected_rows_by_year.values())
assert procurement_df["procurement_id"].is_unique
assert not procurement_df.duplicated().any()
assert procurement_df["source_document"].notna().all()
assert procurement_df["source_url"].notna().all()
assert set(procurement_df["year"].dropna().unique()) == set(expected_rows_by_year)
assert not procurement_df["procurement_id"].str.contains(r"Class|UNSPSC|Page", case=False, na=False).any()

print("Total procurement rows:", len(procurement_df))
print("Total procurement columns:", len(procurement_df.columns))
print("Duplicate IDs:", procurement_df["procurement_id"].duplicated().sum())
print("Duplicate full rows:", procurement_df.duplicated().sum())
print("Years covered:", sorted(procurement_df["year"].dropna().unique().tolist()))
print("\nRows by year:")
print(procurement_df.groupby("year").size())
print("\nMissing values by column:")
print(procurement_df.isna().sum())
print("\nData types:")
print(procurement_df.dtypes)
print("\nSource documents:")
print(procurement_df[["year", "source_document", "source_url"]].drop_duplicates().to_string(index=False))

Total procurement rows: 147
Total procurement columns: 17
Duplicate IDs: 0
Duplicate full rows: 0
Years covered: [2023, 2024, 2025]

Rows by year:
year
2023    47
2024    37
2025    63
dtype: int64

Missing values by column:
procurement_id            0
year                      0
procurement_class         0
unspsc                    0
description             110
reference_number         92
unit_of_measure           1
quantity                 64
source_of_funds           0
procurement_method       37
publication_date          0
award_date               71
start_date                0
estimated_budget_zmw     84
comments                 90
source_document           0
source_url                0
dtype: int64

Data types:
procurement_id           string
year                      Int64
procurement_class        string
unspsc                   string
description              string
reference_number         string
unit_of_measure          string
quantity                Float64
source_of_funds  

## 12. Procurement Export

The clean procurement dataset is written as UTF-8, pipe-separated CSV and read back immediately for a second validation of its row count, column count, unique IDs, and provenance.

In [17]:
procurement_df.to_csv(
    procurement_output_path,
    sep="|",
    index=False,
    encoding="utf-8",
)

procurement_check_df = pd.read_csv(procurement_output_path, sep="|")
assert procurement_check_df.shape == procurement_df.shape
assert procurement_check_df["procurement_id"].is_unique
assert not procurement_check_df.duplicated().any()
assert procurement_check_df["source_url"].notna().all()

print("Exported procurement shape:", procurement_check_df.shape)
print("Exported duplicate IDs:", procurement_check_df["procurement_id"].duplicated().sum())
print("Exported duplicate full rows:", procurement_check_df.duplicated().sum())

Exported procurement shape: (147, 17)
Exported duplicate IDs: 0
Exported duplicate full rows: 0


# PART C - FINANCIAL DATASET

## 13. Financial Source Discovery and Limitation

The official publications page lists audited financial statements, budgets, and a 2025 budget-performance report. The 2024 annual-budget PDF download was incomplete and failed structural validation, so a financial CSV is intentionally not exported in this version. No amounts are inferred from narrative text, and no incomplete financial dataset is created.

## 18. Combined Dataset Summary

Completed outputs are the unchanged 84-record CDF project dataset and the 147-record procurement-plan dataset. Together they connect community-project approvals with council procurement planning over 2023-2025, improving coverage of council service delivery, funding sources, procurement classes, timings, and planned 2025 budget values.

## 19. Limitations

The procurement plans are planning records, not evidence that procurement was completed. The 2023 and 2025 source layouts clip some description/reference cells, so those values are retained as missing rather than guessed. A financial dataset is not included because it has not yet met the same reliability threshold.

## 20. Conclusion

The notebook reproduces two completed official-source datasets from a fresh kernel: the preserved 2025 CDF project dataset and a validated 2023-2025 procurement dataset. Both exports retain year and source provenance and use the required pipe-separated format.